## 🎧 Interview Audio Emotion & Prosody Analysis

<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/extract_video_interviews_audio_emotion_1.wav" type="audio/wav">
</audio>

---



### Speech Affect, Tempo, and Emotional Signaling (Process Digital Twin Agent)

This notebook implements an **audio-based affect analysis agent** as part of the agent-based digital twin described in paper about *An Agent-Based Digital Twin for AI in Student Admissions*.


In high stakes interview settings, *what* a candidate says is inseparable from *how* it is said.
Speech rhythm, pitch dynamics, pauses, and emotional tone provide crucial context for interpreting confidence, stress, authenticity, and cognitive load—signals that are often assessed implicitly by human evaluators but rarely formalized or audited.

This notebook demonstrates how such **paralinguistic cues** can be:

* explicitly measured,
* represented as structured data,
* and integrated as an independent, inspectable agent within a transparent admission workflow.

### What this agent analyzes

For each interview audio segment, the notebook extracts:

**Prosodic features**

* speaking tempo (BPM),
* silence and pause ratios,
* pitch statistics (average and variability),

**Emotional signals**

* probability-weighted emotion labels inferred from speech,
* short-window emotion estimation to reduce noise from long recordings.

Each audio file is processed **independently**, reflecting the parallel-agent design described in the paper.
For reproducibility in Google Colab, these agents are executed sequentially as standalone notebooks.

### Inputs

* Interview audio tracks extracted from video
  (`video_interviews_audio_files/`)

### Outputs

* Structured JSON emotion and prosody profiles
  (`video_interviews_audio_emotions/`)

### Relation to the paper

This notebook operationalizes one of the paper's central claims:
that subjective human judgments—such as *nervousness*, *confidence*, or "unnatural delivery"—can be decomposed into **explicit, measurable signals** handled by specialized agents.

In a production deployment, anonymized audio segments could be analyzed by API-based services or on-premise models.
Here, a fully local and reproducible setup is used to demonstrate **methodological transparency**, enabling reviewers and readers to inspect both the signals and the assumptions behind them.

---


<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/extract_video_interviews_audio_emotion_2.wav" type="audio/wav">
</audio>

In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

In [ ]:
# 1. Install Dependencies
!pip install -q librosa transformers torch numpy scipy

In [ ]:
import os
import json
import numpy as np
import librosa
import torch
from transformers import pipeline
from pathlib import Path
from google.colab import drive

# 2. Mount Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 3. Initialize Models
print("Loading Emotion Recognition Model...")
emotion_classifier = pipeline("audio-classification", model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition")
print("Model loaded.")

Loading Emotion Recognition Model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition
Key                      | Status     | 
-------------------------+------------+-
classifier.output.weight | UNEXPECTED | 
classifier.dense.weight  | UNEXPECTED | 
classifier.dense.bias    | UNEXPECTED | 
classifier.output.bias   | UNEXPECTED | 
classifier.bias          | MISSING    | 
projector.bias           | MISSING    | 
classifier.weight        | MISSING    | 
projector.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# 4. Define Analysis Function
def analyze_audio(file_path):
    results = {}

    try:
        # Load Audio
        # sr=16000 is standard for most DL models
        y, sr = librosa.load(file_path, sr=16000)

        # --- Prosodic Analysis ---

        # Tempo (BPM)
        onset_env = librosa.onset.onset_strength(y=y, sr=sr)
        tempo = librosa.feature.tempo(onset_envelope=onset_env, sr=sr)
        results['tempo_bpm'] = float(tempo[0])

        # Silence / Pauses
        non_silent_intervals = librosa.effects.split(y, top_db=20)
        non_silent_duration = sum([end - start for start, end in non_silent_intervals]) / sr
        total_duration = librosa.get_duration(y=y, sr=sr)
        silence_ratio = (total_duration - non_silent_duration) / total_duration
        results['silence_ratio'] = float(silence_ratio)

        # Pitch (Fundamental Frequency - F0)
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
        valid_f0 = f0[~np.isnan(f0)]

        if len(valid_f0) > 0:
            results['pitch_avg_hz'] = float(np.mean(valid_f0))
            results['pitch_std_hz'] = float(np.std(valid_f0))
        else:
            results['pitch_avg_hz'] = None
            results['pitch_std_hz'] = None

        # --- Emotion Recognition ---

        # Max 30 seconds for emotion model
        max_duration_sec = 30
        chunk_samples = int(max_duration_sec * sr)
        audio_chunk = y[:chunk_samples] if len(y) > chunk_samples else y

        # Predict
        labels = emotion_classifier(audio_chunk, top_k=5)

        # Format labels into a cleaner dictionary
        emotions = {}
        for label in labels:
            emotions[label['label']] = float(label['score'])
        results['emotions'] = emotions

        return results

    except Exception as e:
        print(f"Error analyzing {file_path}: {e}")
        return None

In [ ]:
# 5. File Scanning & Processing Loop

mid_drive_dir = Path("/content/drive/MyDrive/colab-output/dream_applicant_application/")

print(f"Scanning directory: {mid_drive_dir}")

# Iterate through applicant folders
for applicant_dir in mid_drive_dir.iterdir():
    if applicant_dir.is_dir():
        audio_files_dir = applicant_dir / "video_interviews_audio_files"

        if audio_files_dir.exists() and audio_files_dir.is_dir():
            print(f"\nFound audio folder: {audio_files_dir.name} in {applicant_dir.name}")

            # Create output directory for emotions
            emotion_output_dir = applicant_dir / "video_interviews_audio_emotions"
            os.makedirs(emotion_output_dir, exist_ok=True)

            # Helper to process a single mp3 file
            def process_mp3(mp3_file, relative_subdir=None):
                print(f"  Processing: {mp3_file.name}")

                # Determine output path
                if relative_subdir:
                    output_subdir = emotion_output_dir / relative_subdir
                    os.makedirs(output_subdir, exist_ok=True)
                    output_json_path = output_subdir / (mp3_file.stem + "_emotion.json")
                else:
                    output_json_path = emotion_output_dir / (mp3_file.stem + "_emotion.json")

                # Skip if already exists
                if output_json_path.exists():
                    print(f"    -> Skipping (already exists)")
                    return

                # Analyze
                analysis_result = analyze_audio(str(mp3_file))

                if analysis_result:
                    # Save to JSON
                    with open(output_json_path, 'w') as f:
                        json.dump(analysis_result, f, indent=4)
                    print(f"    -> Saved to {output_json_path.name}")

            # 1. Scan direct files
            for item in audio_files_dir.iterdir():
                if item.is_file() and item.suffix.lower() == '.mp3':
                    process_mp3(item)

            # 2. Scan subdirectories
            for item in audio_files_dir.iterdir():
                if item.is_dir():
                    for subfile in item.iterdir():
                        if subfile.is_file() and subfile.suffix.lower() == '.mp3':
                            process_mp3(subfile, relative_subdir=item.name)

print("\n✅ All audio analysis completed!")

Scanning directory: /content/drive/MyDrive/colab-output/dream_applicant_application

Found audio folder: video_interviews_audio_files in 25598_35760
  Processing: What is your life dream in the future as engineer_.mp3
    -> Saved to What is your life dream in the future as engineer__emotion.json
  Processing: What skills and talents makes you fit for Mechatronics Systems studies_.mp3
    -> Saved to What skills and talents makes you fit for Mechatronics Systems studies__emotion.json
  Processing: What are your expectations in the mechatronic engineer profession_.mp3
    -> Saved to What are your expectations in the mechatronic engineer profession__emotion.json
  Processing: What scientific research you plan to perform in program Mechatronics Systems_.mp3
    -> Saved to What scientific research you plan to perform in program Mechatronics Systems__emotion.json

Found audio folder: video_interviews_audio_files in 1000000002_1000000002
  Processing: Who are you.mp3
    -> Saved to Who ar


---

## ▶️ Where to Go Next

This audio emotion and prosody analysis represents an **alternative processing path** in the process digital twin.
It can be used independently, or in parallel with transcript-based analysis, depending on the evaluation strategy.

### Return to Audio Extraction (if needed)

You can return to the audio extraction step:

**Audio Extraction from Interview Videos**
> 👉 [https://colab.research.google.com/drive/1jTYrU0brHLeMTI65ZzKUDjL8iiCGddPp](https://colab.research.google.com/drive/1jTYrU0brHLeMTI65ZzKUDjL8iiCGddPp)

*And continue with the main evaluation path*

Or just go to :

> **Step 3 – Speech Transcription from Interview Audio**
👉 [https://colab.research.google.com/drive/1tXdu3AAVjaYeGjpG8k5sA4Dk2SB9Z-u3](https://colab.research.google.com/drive/1tXdu3AAVjaYeGjpG8k5sA4Dk2SB9Z-u3)



